# 44 · 质量、多样性与多场景目标

单元：S15。先修：35、43。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：构造任务目标；非真实曝光或商业增益。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 场景与约束
信息查询要覆盖多个方面；导航任务优先官方；交易任务可能要求库存。相同候选，不同需求的最优列表不同。相关性与质量分不是同一个标签。

In [2]:
rel={'a':.95,'b':.94,'c':.90,'d':.80}
print(pd.DataFrame([{'id':d,'rel':rel[d],**DOCUMENTS[d]} for d in DOCUMENTS]))

  id   rel                 title                                     body  \
0  a  0.95          apple repair           official phone battery service   
1  b  0.94           apple fruit   battery repair mentioned in fruit spam   
2  c  0.90  phone battery repair          independent apple service guide   
3  d  0.80    apple repair guide  apple phone battery repair instructions   

     topic  quality  
0  service      1.0  
1     food      0.1  
2  service      0.8  
3    guide      0.9  


## 质量过滤
仅改最低质量阈值；报告相关性和低质曝光数，不能把质量提升表述成NDCG必升。

In [3]:
for threshold in [0.,.5]:
    ids=sorted([d for d in DOCUMENTS if DOCUMENTS[d]['quality']>=threshold],key=lambda d:-rel[d])[:2]
    print(threshold,ids,'rel sum',sum(rel[d] for d in ids),'low quality',sum(DOCUMENTS[d]['quality']<.5 for d in ids))

0.0 ['a', 'b'] rel sum 1.89 low quality 1
0.5 ['a', 'c'] rel sum 1.85 low quality 0


## 多样性惩罚
贪心score=rel−lambda×已选同topic个数；这是透明机制，不叫完整MMR/DPP。先去低质，只改lambda，观察覆盖与相关性代价。

In [4]:
def diversify(lam):
    chosen=[]; rest={d for d in DOCUMENTS if DOCUMENTS[d]['quality']>=.5}
    while rest and len(chosen)<2:
        d=min(rest,key=lambda d:(-(rel[d]-lam*sum(DOCUMENTS[d]['topic']==DOCUMENTS[c]['topic'] for c in chosen)),d))
        chosen.append(d); rest.remove(d)
    return chosen
for lam in [0.,.3]:
    ids=diversify(lam); print(lam,ids,'topics',len({DOCUMENTS[d]['topic'] for d in ids}),'rel',sum(rel[d] for d in ids))
assert len({DOCUMENTS[d]['topic'] for d in diversify(.3)})==2

0.0 ['a', 'c'] topics 1 rel 1.85
0.3 ['a', 'd'] topics 2 rel 1.75


## 曝光偏差
相同真实吸引力，位置曝光率不同会制造点击差异。这里已知构造propensity仅解释机制，真实Lab无曝光概率不能声称IPS纠偏。线上需随机实验与任务护栏。

In [5]:
attraction=np.array([.6,.6]); exposure=np.array([1.,.2])
click_rate=attraction*exposure
print('observed clicks',click_rate,'constructed correction',click_rate/exposure)
assert np.allclose(click_rate/exposure,attraction)

observed clicks [0.6  0.12] constructed correction [0.6 0.6]


## 本次保存结果的解读

质量阈值过滤b使top2相关性和从1.89降到1.85，同时低质曝光由1变0；多样性lambda从0改0.3再以0.10相关性代价换来两个topic。构造曝光概率让相同吸引力呈现不同点击率；真实数据缺propensity时不能照搬纠偏结论。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。